### Import Libraries and Reproducability

In [ ]:
import sys, os, re, json, pandas as pd, numpy as np
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
import transformers

SEED = 22
transformers.set_seed(SEED)
rng  = np.random.default_rng(SEED)

### Configuration and Instructions

In [ ]:
TRAIN_PATH = "train_w1.csv"
TEST_PATH  = "test_w1.csv"
LABEL      = "SNAP"
OUTPUT_DIR = "./results_wave1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

DEMO_FEATURES = [
    "marital.status", "sex", "hispanic.origin", "race",
    "education", "citizenship", "employment", "state", "age",
]
INCOME_FEATURES  = DEMO_FEATURES + ["income"]
FULL_FEATURES    = INCOME_FEATURES
MINIMAL_FEATURES = ["income", "employment", "education"]

COL_MAP = {
    "marital.status":  "Marital Status",
    "sex":             "Sex",
    "hispanic.origin": "Hispanic Origin",
    "race":            "Race",
    "education":       "Educational Attainment",
    "citizenship":     "Citizenship",
    "employment":      "Employment Status",
    "state":           "State of Residence",
    "age":             "Age",
    "income":          "Income",
    "SNAP":            "Coverage for SNAP in Wave 1",
}

INSTRUCTION_STRUCTURED = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food-purchasing "
    "assistance to low-income individuals and families in the U.S. "
    "You will be given a respondent's demographic information and income from "
    "Wave 1 of the 2014 Survey of Income and Program Participation (SIPP). "
    "Based on this information, classify if the person has SNAP coverage in Wave 1. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

INSTRUCTION_NARRATIVE = (
    "You are an expert text classifier. "
    "The Supplemental Nutrition Assistance Program (SNAP) provides food assistance "
    "to low-income households in the U.S. "
    "You will read a short profile of a survey respondent from the 2014 Survey of "
    "Income and Program Participation (SIPP), Wave 1. "
    "Based on the profile, decide whether this person has SNAP coverage. "
    "Return only one label: 'Yes' or 'No', without any other text.\n"
)

INSTRUCTION_MINIMAL = INSTRUCTION_STRUCTURED

### Data Loading

In [ ]:
df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train : {len(df_train):,} rows | SNAP distribution: {dict(Counter(df_train[LABEL]))}")
print(f"Test  : {len(df_test):,}  rows | SNAP distribution: {dict(Counter(df_test[LABEL]))}")

Train : 44,907 rows | SNAP distribution: {'No': 39577, 'Yes': 5330}
Test  : 11,226  rows | SNAP distribution: {'No': 9894, 'Yes': 1332}


### Oversampling

In [ ]:
def oversample_minority(df, label_col, seed=SEED):
    majority = df[df[label_col] == "No"]
    minority = df[df[label_col] == "Yes"]
    minority_upsampled = minority.sample(
        n=len(majority), replace=True, random_state=seed
    )
    return pd.concat([majority, minority_upsampled]).sample(
        frac=1, random_state=seed
    ).reset_index(drop=True)

df_train_balanced = oversample_minority(df_train, LABEL)
print(f"\nBalanced train: {len(df_train_balanced):,} rows | "
      f"SNAP distribution: {dict(Counter(df_train_balanced[LABEL]))}")


Balanced train: 79,154 rows | SNAP distribution: {'Yes': 39577, 'No': 39577}


### Prompt Builders

In [ ]:
def build_structured_prompt(row, features, instruction=INSTRUCTION_STRUCTURED):
    user_content = "\n".join(
        f"{COL_MAP[k]}: {row[k]}" for k in features if k in row
    )
    return {
        "prompt":     [{"role": "system", "content": instruction},
                       {"role": "user",   "content": user_content}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }

def build_narrative_prompt(row, features, instruction=INSTRUCTION_NARRATIVE):
    age         = row.get("age", "unknown age")
    sex         = row.get("sex", "").lower()
    hispanic    = row.get("hispanic.origin", "")
    race        = row.get("race", "")
    ethnicity   = f"{hispanic} {race}".strip()
    marital     = row.get("marital.status", "").lower()
    edu         = row.get("education", "").lower()
    employment  = row.get("employment", "").lower()
    state       = row.get("state", "")
    income      = row.get("income", "")
    citizenship = row.get("citizenship", "").lower()

    narrative = (
        f"A {age}-year-old {ethnicity} {sex}, {marital}, residing in {state}. "
        f"Educational attainment: {edu}. "
        f"Citizenship: {citizenship}. "
        f"Employment in the reference month: {employment}. "
        f"Monthly household income: {income}."
    )
    return {
        "prompt":     [{"role": "system", "content": instruction},
                       {"role": "user",   "content": narrative}],
        "completion": [{"role": "assistant", "content": str(row[LABEL])}],
    }

def make_dataset(df, prompt_fn, features):
    from datasets import Dataset
    records = [prompt_fn(row, features) for _, row in df.iterrows()]
    return Dataset.from_list(records)

### Export to JSON

In [ ]:
def export_to_jsonl(df, prompt_fn, features, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            sample   = prompt_fn(row, features)
            messages = sample["prompt"] + sample["completion"]
            f.write(json.dumps({"messages": messages}) + "\n")

experiments_data = {
    "T1A_imbalanced":  (df_train,          build_structured_prompt, FULL_FEATURES,    FULL_FEATURES),
    "T1B_oversampled": (df_train_balanced,  build_structured_prompt, FULL_FEATURES,    FULL_FEATURES),
    "T2A_demo_only":   (df_train,          build_structured_prompt, DEMO_FEATURES,    DEMO_FEATURES),
    "T2C_full_model":  (df_train,          build_structured_prompt, FULL_FEATURES,    FULL_FEATURES),
    "T3B_narrative":   (df_train,          build_narrative_prompt,  FULL_FEATURES,    FULL_FEATURES),
    "T3B_minimal":     (df_train,          build_structured_prompt, MINIMAL_FEATURES, MINIMAL_FEATURES),
}

os.makedirs("./mlx_data", exist_ok=True)
for run_label, (train_df, prompt_fn, train_feats, val_feats) in experiments_data.items():
    folder = f"./mlx_data/{run_label}"
    os.makedirs(folder, exist_ok=True)
    export_to_jsonl(train_df, prompt_fn, train_feats, f"{folder}/train.jsonl")
    export_to_jsonl(df_test,  prompt_fn, val_feats,   f"{folder}/valid.jsonl")
    print(f"Exported {folder}/")

Exported ./mlx_data/T1A_imbalanced/
Exported ./mlx_data/T1B_oversampled/
Exported ./mlx_data/T2A_demo_only/
Exported ./mlx_data/T2C_full_model/
Exported ./mlx_data/T3B_narrative/
Exported ./mlx_data/T3B_minimal/


### Making Test Datasets

In [ ]:
test_ds_structured_full    = make_dataset(df_test, build_structured_prompt, FULL_FEATURES)
test_ds_structured_demo    = make_dataset(df_test, build_structured_prompt, DEMO_FEATURES)
test_ds_narrative_full     = make_dataset(df_test, build_narrative_prompt,  FULL_FEATURES)
test_ds_structured_minimal = make_dataset(df_test, build_structured_prompt, MINIMAL_FEATURES)
print("Test datasets ready.")

Test datasets ready.


### Defining Predictions

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)

def evaluate_predictions(predictions, references, label_name=""):
    valid         = [(p, r) for p, r in zip(predictions, references) if p in ["Yes", "No"]]
    unknown_count = len(predictions) - len(valid)
    if not valid:
        print("No valid predictions.")
        return {}
    y_pred, y_true = zip(*valid)
    metrics = {
        "Label":     label_name,
        "N_test":    len(df_test),
        "Accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "Recall":    round(recall_score(y_true, y_pred, pos_label="Yes"), 4),
        "Precision": round(precision_score(y_true, y_pred, pos_label="Yes", zero_division=0), 4),
        "F1":        round(f1_score(y_true, y_pred, pos_label="Yes"), 4),
        "Unknown":   unknown_count,
    }
    print(f"\n── {label_name} ──")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return metrics

### Calibration Analytics

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

def calibration_analysis(probabilities, references, label_name, save_path):
    y_true_bin = [1 if r == "Yes" else 0 for r in references]
    probs      = np.array(probabilities)

    n_bins    = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece       = 0.0
    for i in range(n_bins):
        mask = (probs >= bin_edges[i]) & (probs < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = np.mean(np.array(y_true_bin)[mask])
        bin_conf = np.mean(probs[mask])
        ece     += mask.sum() / len(probs) * abs(bin_acc - bin_conf)

    frac_pos, mean_pred = calibration_curve(y_true_bin, probs, n_bins=10)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.plot(mean_pred, frac_pos, "s-", label=f"{label_name} (ECE={ece:.4f})")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(f"Reliability Diagram — {label_name}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  ECE: {ece:.4f} | Saved → {save_path}")
    return round(ece, 4)

### Run Inference Function

In [ ]:
from mlx_lm import load, generate

def format_chat_prompt(prompt_list):
    return "".join(
        f"<|{m['role']}|>\n{m['content']}\n" for m in prompt_list
    ) + "<|assistant|>\n"

def run_inference_mlx(model, tokenizer, dataset, return_probs=False):
    predictions, references, probabilities = [], [], []

    for example in tqdm(dataset, desc="Generating"):
        prompt_text = format_chat_prompt(example["prompt"])
        response    = generate(
            model, tokenizer,
            prompt=prompt_text,
            max_tokens=5,
            verbose=False
        )
        match = re.search(r"\b(Yes|No)\b", response, re.IGNORECASE)
        predictions.append(match.group(1).title() if match else "Unknown")
        references.append(example["completion"][0]["content"].strip().title())

        if return_probs:
            if match:
                probabilities.append(1.0 if match.group(1).title() == "Yes" else 0.0)
            else:
                probabilities.append(0.5)

    if return_probs:
        return predictions, references, probabilities
    return predictions, references

### Lora Configuration

In [ ]:
import yaml

lora_config = {
    "lora_parameters": {
        "rank": 16,
        "scale": 2.0,   # equivalent to alpha=32 with rank=16 (scale = alpha/rank = 32/16)
        "dropout": 0.05,
    }
}

os.makedirs("./configs", exist_ok=True)
with open("./configs/lora_config.yaml", "w") as f:
    yaml.dump(lora_config, f)

print("LoRA config saved to ./configs/lora_config.yaml")

LoRA config saved to ./configs/lora_config.yaml


### Training and Inference for T1A_imbalanced

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T1A_imbalanced \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T1A_imbalanced \
  --save-every 500 \
  -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|███████████████████████████| 9/9 [00:17<00:00,  1.97s/it]
Download complete: : 880MB [00:17, 49.7MB/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:21<00:00,  1.15it/s]
Iter 1: Val loss 7.769, Val took 21.796s
Iter 10: Train loss 4.020, Learning Rate 2.000e-04, It/sec 0.633, Tokens/sec 10.130, Trained Tokens 160, Peak mem 5.275 GB
Iter 20: Train loss 0.173, Learning Rate 2.000e-04, It/sec 0.699, Tokens/sec 11.176, Trained Tokens 320, Peak mem 5.275 GB
Iter 30: Train loss 0.111, Learning Rate 2.000e-04, It/sec 0.696, Tokens/sec 11.133, Trained Tokens 480, Peak mem 5.275 GB
Iter 40: Train loss 0.058, Learning Rate 2.000e-04, It/sec 0.699, Tokens/sec 11.182, Trained Tokens 640, Peak mem 5.275 GB
Iter 50: Train loss 0.076, Learning Rate 2.000e-04, It/sec

In [ ]:
import time

all_metrics = []
experiment_times = {}

# ── Load model with T1-A adapter ─────────────────────────────
model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T1A_imbalanced",
)

# ── Inference ─────────────────────────────────────────────────
t0 = time.time()
preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

# ── Evaluate ──────────────────────────────────────────────────
metrics = evaluate_predictions(preds, refs, label_name="T1A_imbalanced")

# ── Calibration ───────────────────────────────────────────────
ece = calibration_analysis(
    probs, refs,
    label_name="T1A_imbalanced",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T1A_imbalanced.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

# ── Save predictions ──────────────────────────────────────────
out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1A_imbalanced.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1A_imbalanced"] = elapsed
print(f"\n  ⏱ T1A_imbalanced done in {elapsed:.1f}s")

# ── Clean up ──────────────────────────────────────────────────
del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11226 [00:00<?, ?it/s]


── T1A_imbalanced ──
  Label: T1A_imbalanced
  N_test: 11226
  Accuracy: 0.8909
  Recall: 0.2117
  Precision: 0.6171
  F1: 0.3153
  Unknown: 0
              precision    recall  f1-score   support

          No       0.90      0.98      0.94      9894
         Yes       0.62      0.21      0.32      1332

    accuracy                           0.89     11226
   macro avg       0.76      0.60      0.63     11226
weighted avg       0.87      0.89      0.87     11226

  ECE: 0.0935 | Saved → ./results_wave2/calibration_T1A_imbalanced.png

  ⏱ T1A_imbalanced done in 4002.7s


### Training and Inference for T1B_oversampled

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T1B_oversampled \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T1B_oversampled \
  --save-every 500 \
  -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 73014.96it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:19<00:00,  1.30it/s]
Iter 1: Val loss 7.676, Val took 19.175s
Iter 10: Train loss 4.534, Learning Rate 2.000e-04, It/sec 0.626, Tokens/sec 10.012, Trained Tokens 160, Peak mem 5.275 GB
Iter 20: Train loss 0.421, Learning Rate 2.000e-04, It/sec 0.690, Tokens/sec 11.042, Trained Tokens 320, Peak mem 5.275 GB
Iter 30: Train loss 0.410, Learning Rate 2.000e-04, It/sec 0.695, Tokens/sec 11.117, Trained Tokens 480, Peak mem 5.275 GB
Iter 40: Train loss 0.424, Learning Rate 2.000e-04, It/sec 0.691, Tokens/sec 11.055, Trained Tokens 640, Peak mem 5.275 GB
Iter 50: Train loss 3.755, Learning Rate 2.000e-04, It/sec 0.6

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T1B_oversampled",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T1B_oversampled")

ece = calibration_analysis(
    probs, refs,
    label_name="T1B_oversampled",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T1B_oversampled.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T1B_oversampled.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T1B_oversampled"] = elapsed
print(f"\n  ⏱ T1B_oversampled done in {elapsed:.1f}s")

del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11226 [00:00<?, ?it/s]


── T1B_oversampled ──
  Label: T1B_oversampled
  N_test: 11226
  Accuracy: 0.8608
  Recall: 0.4617
  Precision: 0.4209
  F1: 0.4404
  Unknown: 0
              precision    recall  f1-score   support

          No       0.93      0.91      0.92      9894
         Yes       0.42      0.46      0.44      1332

    accuracy                           0.86     11226
   macro avg       0.67      0.69      0.68     11226
weighted avg       0.87      0.86      0.86     11226

  ECE: 0.0639 | Saved → ./results_wave2/calibration_T1B_oversampled.png

  ⏱ T1B_oversampled done in 3357.1s


### Training and Inference for T2A_demo_only

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T2A_demo_only \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T2A_demo_only \
  --save-every 500 \
  -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 87381.33it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:16<00:00,  1.50it/s]
Iter 1: Val loss 7.528, Val took 16.615s
Iter 10: Train loss 3.913, Learning Rate 2.000e-04, It/sec 0.759, Tokens/sec 12.141, Trained Tokens 160, Peak mem 5.178 GB
Iter 20: Train loss 0.217, Learning Rate 2.000e-04, It/sec 0.805, Tokens/sec 12.881, Trained Tokens 320, Peak mem 5.178 GB
Iter 30: Train loss 0.196, Learning Rate 2.000e-04, It/sec 0.794, Tokens/sec 12.712, Trained Tokens 480, Peak mem 5.233 GB
Iter 40: Train loss 0.050, Learning Rate 2.000e-04, It/sec 0.807, Tokens/sec 12.905, Trained Tokens 640, Peak mem 5.233 GB
Iter 50: Train loss 0.061, Learning Rate 2.000e-04, It/sec 0.7

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T2A_demo_only",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_demo, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T2A_demo_only")

ece = calibration_analysis(
    probs, refs,
    label_name="T2A_demo_only",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T2A_demo_only.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2A_demo_only.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2A_demo_only"] = elapsed
print(f"\n  ⏱ T2A_demo_only done in {elapsed:.1f}s")

del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11226 [00:00<?, ?it/s]


── T2A_demo_only ──
  Label: T2A_demo_only
  N_test: 11226
  Accuracy: 0.886
  Recall: 0.0908
  Precision: 0.6368
  F1: 0.159
  Unknown: 0
              precision    recall  f1-score   support

          No       0.89      0.99      0.94      9894
         Yes       0.64      0.09      0.16      1332

    accuracy                           0.89     11226
   macro avg       0.76      0.54      0.55     11226
weighted avg       0.86      0.89      0.85     11226

  ECE: 0.1079 | Saved → ./results_wave2/calibration_T2A_demo_only.png

  ⏱ T2A_demo_only done in 3426.6s


### Training and Inference for T2C_full_model

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T2C_full_model \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T2C_full_model \
  --save-every 500 \
  -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|███████████████████████| 9/9 [00:00<00:00, 118334.60it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:18<00:00,  1.34it/s]
Iter 1: Val loss 7.769, Val took 18.662s
Iter 10: Train loss 4.020, Learning Rate 2.000e-04, It/sec 0.689, Tokens/sec 11.031, Trained Tokens 160, Peak mem 5.275 GB
Iter 20: Train loss 0.173, Learning Rate 2.000e-04, It/sec 0.692, Tokens/sec 11.070, Trained Tokens 320, Peak mem 5.275 GB
Iter 30: Train loss 0.111, Learning Rate 2.000e-04, It/sec 0.667, Tokens/sec 10.679, Trained Tokens 480, Peak mem 5.275 GB
Iter 40: Train loss 0.058, Learning Rate 2.000e-04, It/sec 0.671, Tokens/sec 10.728, Trained Tokens 640, Peak mem 5.275 GB
Iter 50: Train loss 0.076, Learning Rate 2.000e-04, It/sec 0.6

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T2C_full_model",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T2C_full_model")

ece = calibration_analysis(
    probs, refs,
    label_name="T2C_full_model",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T2C_full_model.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T2C_full_model.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T2C_full_model"] = elapsed
print(f"\n  ⏱ T2C_full_model done in {elapsed:.1f}s")

del model_mlx, tok_mlx

### Inference for Zero-shot model

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    # no adapter_path — base model only
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T3A_zeroshot")

ece = calibration_analysis(
    probs, refs,
    label_name="T3A_zeroshot",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T3A_zeroshot.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3A_zeroshot.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3A_zeroshot"] = elapsed
print(f"\n  ⏱ T3A_zeroshot done in {elapsed:.1f}s")

del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11226 [00:00<?, ?it/s]


── T3A_zeroshot ──
  Label: T3A_zeroshot
  N_test: 11226
  Accuracy: 0.8805
  Recall: 0.0015
  Precision: 0.1429
  F1: 0.003
  Unknown: 0
              precision    recall  f1-score   support

          No       0.88      1.00      0.94      9894
         Yes       0.14      0.00      0.00      1332

    accuracy                           0.88     11226
   macro avg       0.51      0.50      0.47     11226
weighted avg       0.79      0.88      0.83     11226

  ECE: 0.1185 | Saved → ./results_wave2/calibration_T3A_zeroshot.png

  ⏱ T3A_zeroshot done in 2899.9s


### Training and Inference for T3B_narrative

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T3B_narrative \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T3B_narrative \
  --save-every 500 \
  -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 97541.95it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:16<00:00,  1.55it/s]
Iter 1: Val loss 7.609, Val took 16.128s
Iter 10: Train loss 3.923, Learning Rate 2.000e-04, It/sec 0.795, Tokens/sec 12.721, Trained Tokens 160, Peak mem 4.790 GB
Iter 20: Train loss 0.173, Learning Rate 2.000e-04, It/sec 0.821, Tokens/sec 13.137, Trained Tokens 320, Peak mem 4.790 GB
Iter 30: Train loss 0.130, Learning Rate 2.000e-04, It/sec 0.821, Tokens/sec 13.136, Trained Tokens 480, Peak mem 4.790 GB
Iter 40: Train loss 0.075, Learning Rate 2.000e-04, It/sec 0.822, Tokens/sec 13.147, Trained Tokens 640, Peak mem 4.790 GB
Iter 50: Train loss 0.085, Learning Rate 2.000e-04, It/sec 0.8

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T3B_narrative",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_narrative_full, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T3B_narrative")

ece = calibration_analysis(
    probs, refs,
    label_name="T3B_narrative",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T3B_narrative.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3B_narrative.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3B_narrative"] = elapsed
print(f"\n  ⏱ T3B_narrative done in {elapsed:.1f}s")

del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11226 [00:00<?, ?it/s]


── T3B_narrative ──
  Label: T3B_narrative
  N_test: 11226
  Accuracy: 0.8857
  Recall: 0.286
  Precision: 0.5344
  F1: 0.3726
  Unknown: 0
              precision    recall  f1-score   support

          No       0.91      0.97      0.94      9894
         Yes       0.53      0.29      0.37      1332

    accuracy                           0.89     11226
   macro avg       0.72      0.63      0.65     11226
weighted avg       0.87      0.89      0.87     11226

  ECE: 0.0847 | Saved → ./results_wave2/calibration_T3B_narrative.png

  ⏱ T3B_narrative done in 2923.7s


### Training and Inference for T3B_minimal

In [ ]:
!mlx_lm.lora \
  --model mlx-community/Qwen2.5-1.5B-Instruct-4bit \
  --train \
  --data ./mlx_data/T3B_minimal \
  --num-layers 16 \
  --batch-size 4 \
  --grad-accumulation-steps 4 \
  --iters 2000 \
  --learning-rate 2e-4 \
  --mask-prompt \
  --adapter-path ./adapters_w1/T3B_minimal \
  --save-every 500 \
  -c ./configs/lora_config.yaml

Loading configuration file ./configs/lora_config.yaml
Loading pretrained model
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 65649.98it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading datasets
Training
Trainable parameters: 0.684% (10.551M/1543.714M)
Starting training..., iters: 2000
Calculating loss...: 100%|██████████████████████| 25/25 [00:14<00:00,  1.68it/s]
Iter 1: Val loss 7.439, Val took 14.874s
Iter 10: Train loss 4.366, Learning Rate 2.000e-04, It/sec 0.882, Tokens/sec 14.113, Trained Tokens 160, Peak mem 4.790 GB
Iter 20: Train loss 0.489, Learning Rate 2.000e-04, It/sec 0.860, Tokens/sec 13.754, Trained Tokens 320, Peak mem 4.790 GB
Iter 30: Train loss 0.110, Learning Rate 2.000e-04, It/sec 0.897, Tokens/sec 14.357, Trained Tokens 480, Peak mem 4.790 GB
Iter 40: Train loss 0.044, Learning Rate 2.000e-04, It/sec 0.863, Tokens/sec 13.812, Trained Tokens 640, Peak mem 4.790 GB
Iter 50: Train loss 0.121, Learning Rate 2.000e-04, It/sec 0.8

In [ ]:
t0 = time.time()

model_mlx, tok_mlx = load(
    "mlx-community/Qwen2.5-1.5B-Instruct-4bit",
    adapter_path="./adapters_w1/T3B_minimal",
)

preds, refs, probs = run_inference_mlx(
    model_mlx, tok_mlx, test_ds_structured_minimal, return_probs=True
)

metrics = evaluate_predictions(preds, refs, label_name="T3B_minimal")

ece = calibration_analysis(
    probs, refs,
    label_name="T3B_minimal",
    save_path=os.path.join(OUTPUT_DIR, "calibration_T3B_minimal.png")
)
metrics["ECE"] = ece
all_metrics.append(metrics)

out_df = pd.DataFrame({"Predicted_Label": preds, "Ground_Truth": refs, "P_Yes": probs})
out_df.to_csv(os.path.join(OUTPUT_DIR, "predictions_T3B_minimal.csv"), index=False)

elapsed = round(time.time() - t0, 2)
experiment_times["T3B_minimal"] = elapsed
print(f"\n  ⏱ T3B_minimal done in {elapsed:.1f}s")

del model_mlx, tok_mlx

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Generating:   0%|          | 0/11226 [00:00<?, ?it/s]


── T3B_minimal ──
  Label: T3B_minimal
  N_test: 11226
  Accuracy: 0.8814
  Recall: 0.0068
  Precision: 0.5294
  F1: 0.0133
  Unknown: 0
              precision    recall  f1-score   support

          No       0.88      1.00      0.94      9894
         Yes       0.53      0.01      0.01      1332

    accuracy                           0.88     11226
   macro avg       0.71      0.50      0.48     11226
weighted avg       0.84      0.88      0.83     11226

  ECE: 0.1179 | Saved → ./results_wave2/calibration_T3B_minimal.png

  ⏱ T3B_minimal done in 3013.3s


In [ ]:
print(f"Experiments collected: {len(all_metrics)}")
for m in all_metrics:
    print(f"  {m['Label']} — F1: {m['F1']}")

Experiments collected: 6
  T1A_imbalanced — F1: 0.3153
  T1B_oversampled — F1: 0.4404
  T2A_demo_only — F1: 0.159
  T3A_zeroshot — F1: 0.003
  T3B_narrative — F1: 0.3726
  T3B_minimal — F1: 0.0133


### Summary Table

In [ ]:
import pandas as pd
import os

# ── Build summary dataframe from accumulated metrics ──────────
summary_df = pd.DataFrame(all_metrics)

# Reorder columns cleanly
col_order = ["Label", "N_test", "Accuracy", "Recall", "Precision", "F1", "ECE", "Unknown"]
summary_df = summary_df[[c for c in col_order if c in summary_df.columns]]

# ── Save to CSV ───────────────────────────────────────────────
summary_path = os.path.join(OUTPUT_DIR, "llm_wave1_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved → {summary_path}\n")

# ── Save runtimes ─────────────────────────────────────────────
runtime_df = pd.DataFrame(
    list(experiment_times.items()), columns=["Experiment", "Runtime_seconds"]
)
runtime_df["Runtime_minutes"] = (runtime_df["Runtime_seconds"] / 60).round(2)
runtime_path = os.path.join(OUTPUT_DIR, "runtimes_wave1.csv")
runtime_df.to_csv(runtime_path, index=False)
print(f"Runtimes saved → {runtime_path}\n")

# ── Pretty print ──────────────────────────────────────────────
print("=" * 70)
print("WAVE 1 — FULL RESULTS SUMMARY")
print("=" * 70)
print(summary_df.to_string(index=False))
print("\nRUNTIMES")
print(runtime_df.to_string(index=False))

Summary saved → ./results_wave2/llm_wave2_summary.csv

Runtimes saved → ./results_wave2/runtimes_wave2.csv

WAVE 2 — FULL RESULTS SUMMARY
          Label  N_test  Accuracy  Recall  Precision     F1    ECE  Unknown
 T1A_imbalanced   11226    0.8909  0.2117     0.6171 0.3153 0.0935        0
T1B_oversampled   11226    0.8608  0.4617     0.4209 0.4404 0.0639        0
  T2A_demo_only   11226    0.8860  0.0908     0.6368 0.1590 0.1079        0
   T3A_zeroshot   11226    0.8805  0.0015     0.1429 0.0030 0.1185        0
  T3B_narrative   11226    0.8857  0.2860     0.5344 0.3726 0.0847        0
    T3B_minimal   11226    0.8814  0.0068     0.5294 0.0133 0.1179        0

RUNTIMES
     Experiment  Runtime_seconds  Runtime_minutes
 T1A_imbalanced          4002.71            66.71
T1B_oversampled          3357.14            55.95
  T2A_demo_only          3426.60            57.11
   T3A_zeroshot          2899.91            48.33
  T3B_narrative          2923.74            48.73
    T3B_minimal     